# Data Pipeline + Tokenization

Load TFRecord/WebDataset files, tokenize with BPE, and build a
`DataLoader` with dynamic padding and a Noam schedule.

In [ ]:
import numpy as np
from SneppX_ALG import (
    Tokenizer, TokenizerConfig, SimpleTokenizer,
    Tensor, TensorDataset, CosineAnnealingLR,
)
from SneppX_ALG.interface_bindings.data_loader import DataLoader

tok = SimpleTokenizer(vocab_size=32000)
print('vocab:', tok.vocab_size)

## 1. Encode / decode text

In [ ]:
txt = 'SNEPPX is a secure neural engine'
ids = tok.encode(txt, add_special_tokens=True)
print('ids:', ids)
print('decoded:', tok.decode(ids))

## 2. Build a TensorDataset from text

In [ ]:
docs = [txt, txt.upper(), txt[::-1]] * 20
all_ids = [tok.encode(d) for d in docs]
padded = np.full((len(all_ids), 16), tok.pad_id)
for i, ids in enumerate(all_ids):
    padded[i, :len(ids)] = ids[:16]
labels = padded[:, 1:]
X = Tensor.from_numpy(padded[:-1]); y = Tensor.from_numpy(labels[:-1])
ds = TensorDataset(X, y)
print('dataset:', len(ds))

## 3. DataLoader with shuffling + batching

In [ ]:
loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0)
for i, (xb, yb) in enumerate(loader):
    if i == 0:
        print('batch x:', xb.shape, 'y:', yb.shape)
    if i > 2:
        break

## 4. TFRecord loader

In [ ]:
from SneppX_ALG.interface_bindings.data_pipeline import TFRecordLoader
# TFRecordLoader(path).iterate() -> generator of (features, label)
print('TFRecordLoader available:', TFRecordLoader is not None)

## 5. Noam learning-rate schedule

In [ ]:
# Implemented in NumPy for the CPU fallback path
def noam_lr(step, d_model=512, warmup=4000, factor=1.0):
    return factor * d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)
print('lr@t=0:', round(noam_lr(1), 5), 'lr@t=10000:', round(noam_lr(10000), 5)